In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import cv2
import xml.etree.ElementTree as ET
import numpy as np
import random
import pandas as pd
import logging
import shutilimport pandas as pd

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LOG_DIR = "/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/GRAZPED_preprocessed_3000/logs"
os.makedirs(LOG_DIR, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(LOG_DIR, "GRAZPED_preprocessing.log"),
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="w"
)

def log_issue(img_file, message):
    logging.warning(f"{img_file} - {message}")

part1 = "/content/drive/MyDrive/thesis doc/GRAZPEDWRI-DX/images_part1"
annotations_folder = "/content/drive/MyDrive/thesis doc/GRAZPEDWRI-DX/folder_structure/pascalvoc"
output_images_folder = "/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/GRAZPED_preprocessed_3000/images"
output_masks_folder = "/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/GRAZPED_preprocessed_3000/masks"

metadata_principal_path = "/content/drive/MyDrive/thesis doc/GRAZPEDWRI-DX/metadata_principal.csv"

MAX_IMAGES = 3000  # nombre d'images à extraire

os.makedirs(output_images_folder, exist_ok=True)
os.makedirs(output_masks_folder, exist_ok=True)
metadata_main = pd.read_csv(metadata_principal_path)

def parse_pascal_voc(xml_file, allowed_labels=None):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    objects = []
    for obj in root.findall("object"):
        label = obj.find("name").text.strip().lower()
        if allowed_labels is not None and label not in allowed_labels:
            continue
        polygon = []
        for pt in obj.findall("polygon/pt"):
            try:
                x = int(float(pt.find("x").text))
                y = int(float(pt.find("y").text))
                polygon.append([x, y])
            except:
                continue
        if polygon:
            objects.append({"type": "polygon", "points": np.array(polygon, dtype=np.int32), "label": label})
        else:
            bbox = obj.find("bndbox")
            if bbox is not None:
                xmin = int(float(bbox.find("xmin").text))
                ymin = int(float(bbox.find("ymin").text))
                xmax = int(float(bbox.find("xmax").text))
                ymax = int(float(bbox.find("ymax").text))
                objects.append({"type": "bbox", "bbox": [xmin, ymin, xmax, ymax], "label": label})
    return objects

all_images = [f for f in os.listdir(part1) if f.endswith(".png")]
selected_images = random.sample(all_images, min(MAX_IMAGES, len(all_images)))
print(f"Nombre d'images sélectionnées : {len(selected_images)}")

selected_metadata = []
total_images = 0

for idx, img_name in enumerate(selected_images, start=1):
    img_path = os.path.join(part1, img_name)
    xml_path = os.path.join(annotations_folder, img_name.replace(".png", ".xml"))
    if not os.path.exists(xml_path):
        log_issue(img_name, "Pas d'annotation")
        continue

    image = cv2.imread(img_path)
    if image is None:
        log_issue(img_name, "Impossible de lire l'image")
        continue

    h, w = image.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    objects = parse_pascal_voc(xml_path, allowed_labels=["fracture"])
    if not objects:
        log_issue(img_name, "Pas d'objet fracture détecté")
        continue

    for obj in objects:
        if obj["type"] == "polygon":
            pts = obj["points"].tolist()
            pts_clamped = [[max(0, min(w-1, x)), max(0, min(h-1, y))] for x, y in pts]
            pts_clamped = np.array(pts_clamped, dtype=np.int32)
            cv2.fillPoly(mask, [pts_clamped], 255)
        elif obj["type"] == "bbox":
            xmin, ymin, xmax, ymax = obj["bbox"]
            xmin = max(0, min(w-1, xmin))
            ymin = max(0, min(h-1, ymin))
            xmax = max(0, min(w-1, xmax))
            ymax = max(0, min(h-1, ymax))
            cv2.rectangle(mask, (xmin, ymin), (xmax, ymax), 255, -1)

    if mask.shape[:2] != image.shape[:2]:
        mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    if mask.sum() == 0:
        log_issue(img_name, "Masque vide - image ignorée")
        continue

    out_img_name = f"GRAZPED_{idx:06d}.png"
    out_mask_name = f"GRAZPED_{idx:06d}_mask.png"

    cv2.imwrite(os.path.join(output_images_folder, out_img_name), image)
    cv2.imwrite(os.path.join(output_masks_folder, out_mask_name), mask)

    filestem = os.path.splitext(img_name)[0]
    row = metadata_main[metadata_main['filestem'] == filestem]
    if not row.empty:
        row_dict = row.iloc[0].to_dict()
        row_dict.update({
            "filename": out_img_name,
            "mask": out_mask_name,
            "class": "fracture"
        })
        selected_metadata.append(row_dict)

    total_images += 1
    print(f"[OK] {img_name} traité -> {out_img_name}")

selected_metadata_df = pd.DataFrame(selected_metadata)
selected_metadata_csv = os.path.join(output_images_folder, "metadata_selected.csv")
selected_metadata_df.to_csv(selected_metadata_csv, index=False)

print(f"Prétraitement terminé avec succès ! ({total_images} images traitées)")
print(f"Métadonnées sauvegardées dans : {selected_metadata_csv}")


📊 Nombre d'images :
images_part1 : 5031


Céation de CSV

In [ ]:

dataset_csv = "/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/dataset.csv"  # CSV principal
images_folder = "/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/GRAZPED_preprocessed_3000/images"
output_csv = "/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/GRAZPED_preprocessed_3000/final_metadata.csv"

metadata_main = pd.read_csv(dataset_csv)

all_images = [f for f in os.listdir(images_folder) if f.endswith(".png") and not f.endswith("_mask.png")]
all_images.sort()

final_metadata = []

for idx, img_name in enumerate(all_images, start=1):
    filestem = img_name.replace("GRAZPED_", "").replace(".png", "")

    if idx-1 < len(metadata_main):
        row_dict = metadata_main.iloc[idx-1].to_dict()
        row_dict.update({
            "image_file": img_name,
            "mask_file": img_name.replace(".png", "_mask.png"),
            "class": "fracture"
        })
        final_metadata.append(row_dict)

df_final = pd.DataFrame(final_metadata)
df_final.to_csv(output_csv, index=False)
print(f"CSV final créé avec {len(df_final)} images sélectionnées :")
print(output_csv)


CSV final créé avec 2017 images sélectionnées :
/content/drive/MyDrive/thesis doc/GRAZPED_preprocessed/GRAZPED_preprocessed_3000/final_metadata.csv
